In [ ]:
import nltk
nltk.download('all')

In [ ]:
import spacy
nlp=spacy.load('en_core_web_sm')
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize,sent_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud,STOPWORDS

Data Cleaning:

Remove duplicates, handle missing reviews if any, preprocess text (lowercasing, stopwords removal).

In [ ]:
df=pd.read_csv('amazonreviews.tsv', sep='\t')

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['len']=df['review'].apply(len)

In [ ]:
#df['review'] = df['review'].apply(data_clean)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.label.value_counts().plot(kind='bar')

In [ ]:
df.hist('len',by='label',bins=100)

In [ ]:
df.len.describe()

Exploratory Analysis:

 Word clouds, sentiment distribution, most common positive/negative words.

In [ ]:
df['label'] = df['label'].map({'pos': 1, 'neg': 0})

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

pos_text = df[df['label'] == 1]['review'].str.cat(sep=' ')
neg_text = df[df['label'] == 0]['review'].str.cat(sep=' ')

wc_pos = WordCloud(background_color='black').generate(pos_text)
wc_neg = WordCloud(background_color='white').generate(neg_text)

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(wc_pos)
plt.title("Positive")
plt.savefig('wc_pos.jpeg')

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(wc_neg)
plt.title("Negative")
plt.savefig('wc_neg.jpeg')
plt.show()

In [ ]:
print(len(pos_text))
print(len(neg_text))

In [ ]:
df.loc[100,'review']

Model Development:

 Use NLP techniques (TF-IDF, Word2Vec, or BERT embeddings) with models like Logistic Regression, SVM, or Neural Networks.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [ ]:
# Features & Target
X = df['review']
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# TF-IDF
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

Word2Vec

In [ ]:
!pip install gensim

In [ ]:
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
import gensim.downloader


In [ ]:
df1=df['review'].apply(data_clean)

In [ ]:
df1.head()

In [ ]:
# This cell is no longer needed as df1 directly contains the tokenized reviews for Word2Vec input.

In [ ]:
print(df2)

In [ ]:
print(len(df2))

In [ ]:
data=[word_tokenize(words) for words in df2]

In [ ]:
model=Word2Vec(data,min_count=1,sg=0,window=5,vector_size=100,epochs=10)

In [ ]:
model.wv.get_vector('Excellent')

Logistic Regression Model

In [ ]:
import keras
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization,Embedding
from sklearn.metrics import accuracy_score,precision_recall_fscore_support

In [ ]:
def calculate_results(true,pred):
    model_accuracy=accuracy_score(true,pred)
    precision,recall,f1_sc,_= precision_recall_fscore_support(true,pred,average='weighted')
    model_results={
        'accuracy':model_accuracy,
        'precision':precision,
        'recall':recall,
        'f1_score':f1_sc
    }
    return model_results

In [ ]:
# Logistic Regression Model
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred= model.predict(X_test_tfidf)
y_pred

In [ ]:
calculate_results(y_pred,y_test)

SVM model

In [ ]:
# SVM model
from sklearn.svm import SVC
svm_model = SVC()
svm_model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred=svm_model.predict(X_test_tfidf)
y_pred

In [ ]:
calculate_results(y_pred,y_test)

LSTM

In [ ]:
import keras
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization,Embedding
from sklearn.metrics import accuracy_score,precision_recall_fscore_support

In [ ]:
vect= TextVectorization(max_tokens=10000,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    ngrams=None,
    output_mode="int",
    output_sequence_length=15)

In [ ]:
emb=Embedding(input_dim=10000,output_dim=128,input_shape=15)

In [ ]:
train_text,val_text,train_lab,val_lab=train_test_split(X_train.to_numpy(),y_train.to_numpy(),test_size=0.2,random_state=100)

In [ ]:
train_text.shape,val_text.shape,train_lab.shape,val_lab.shape

In [ ]:
lstm= tf.keras.models.Sequential()
lstm.add(tf.keras.Input(shape=(1,),dtype=tf.string))
lstm.add(vect)
vect.adapt(train_text)
lstm.add(emb)
lstm.add(tf.keras.layers.LSTM(units=100,return_sequences=True)) #### First Hidden layer
lstm.add(tf.keras.layers.LSTM(units=100,return_sequences=True))  #### Second Hidden layer
lstm.add(tf.keras.layers.LSTM(units=100))   #### Third Hidden layer
lstm.add(tf.keras.layers.Dense(1,activation='sigmoid'))  #### output layer
lstm.summary()

In [ ]:
#### Model Training and Validation
lstm.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
lstm.fit(train_text,train_lab,epochs=10,validation_data=(val_text,val_lab),batch_size=50)

In [ ]:
pred=tf.squeeze(np.round(lstm.predict(val_text)))

In [ ]:
calculate_results(val_lab,pred)

Validation:

Use train/test split, cross-validation, and metrics like accuracy, F1-score.

I built a sentiment analysis model using TF-IDF features and compared Logistic Regression, SVM, and Neural Networks. SVM achieved the best performance with 87.55% accuracy and balanced precision-recall, making it the most reliable model. Logistic Regression also performed well as a baseline, while the Neural Network underperformed due to lack of tuning and the sparse nature of TF-IDF features.